# Yelp EDA — `yelp_clean.csv`
Goal: verify the data is clean and ML-ready for the restaurant song recommendation engine.

Checks:
1. Shape & dtypes
2. Missing values & `unknown` sentinels
3. Duplicates
4. Stars & review_count distributions
5. Binary column validation
6. Categorical column value counts
7. `unknown` proportion per column
8. Distributions
9. Correlation among ambience flags
10. Cleaning — encode categoricals, cast types, export

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

FILEPATH = r"C:\Users\ryanm\Documents\coding_projects\restaurant_recommendation\data\raw\yelp_clean.csv"

df = pd.read_csv(FILEPATH)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

## 1. Shape & Data Types

In [ ]:
print(df.dtypes)
df.head(3)

## 2. Missing Values & `unknown` Sentinels
No actual NaNs in this dataset, but `NoiseLevel`, `Alcohol`, and `RestaurantsAttire` use the string `"unknown"` as a missing-value sentinel.

In [ ]:
# True NaN check
null_counts = df.isnull().sum()
if null_counts.sum() == 0:
    print("No NaN values found.")
else:
    display(null_counts[null_counts > 0])

In [ ]:
# Columns that use 'unknown' as a sentinel
CATEGORICAL_COLS = ['NoiseLevel', 'Alcohol', 'RestaurantsAttire']

print("'unknown' counts per categorical column:")
for col in CATEGORICAL_COLS:
    n = (df[col] == 'unknown').sum()
    pct = n / len(df) * 100
    print(f"  {col}: {n:,} ({pct:.1f}%)")

## 3. Duplicates

In [ ]:
full_dups = df.duplicated().sum()
print(f"Full duplicate rows: {full_dups}")

id_dups = df.duplicated(subset='business_id').sum()
print(f"Duplicate business_ids: {id_dups}")

if id_dups > 0:
    display(df[df.duplicated(subset='business_id', keep=False)].sort_values('business_id').head(10))

## 4. Stars & review_count

In [ ]:
# Stars should be in {1.0, 1.5, 2.0, ..., 5.0}
valid_stars = {1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0}
invalid_stars = df[~df['stars'].isin(valid_stars)]
print(f"Stars range: {df['stars'].min()} – {df['stars'].max()}")
print(f"Invalid star values: {len(invalid_stars)}")
print("\nStar distribution:")
print(df['stars'].value_counts().sort_index())

In [ ]:
print("review_count summary:")
print(df['review_count'].describe())

# Flag suspiciously low or high review counts
zero_reviews = (df['review_count'] == 0).sum()
high_reviews = (df['review_count'] > 5000).sum()
print(f"\nBusinesses with 0 reviews: {zero_reviews}")
print(f"Businesses with >5000 reviews: {high_reviews}")
if high_reviews > 0:
    display(df[df['review_count'] > 5000][['name', 'city', 'stars', 'review_count']].sort_values('review_count', ascending=False).head(10))

## 5. Binary Column Validation
All binary columns should contain only 0.0 and 1.0.

In [ ]:
BINARY_COLS = [
    'Ambience.romantic', 'Ambience.divey', 'Ambience.classy', 'Ambience.hipster',
    'Ambience.trendy', 'Ambience.upscale', 'Ambience.casual',
    'HasTV', 'RestaurantsGoodForGroups', 'HappyHour',
    'GoodForMeal.breakfast', 'GoodForMeal.brunch', 'GoodForMeal.latenight', 'GoodForMeal.dinner',
    'RestaurantsTableService'
]

violations = {}
for col in BINARY_COLS:
    bad = df[~df[col].isin([0.0, 1.0])]
    if len(bad) > 0:
        violations[col] = bad[col].unique().tolist()

if violations:
    print("Binary violations:")
    for k, v in violations.items():
        print(f"  {k}: unexpected values {v}")
else:
    print(f"All {len(BINARY_COLS)} binary columns contain only 0.0 / 1.0.")

In [ ]:
# Class balance — what % of restaurants have each binary attribute
balance = df[BINARY_COLS].mean().sort_values(ascending=False)
print("Proportion of restaurants with each binary attribute (1 = True):")
print(balance.round(3).to_string())

## 6. Categorical Column Value Counts

In [ ]:
for col in CATEGORICAL_COLS:
    print(f"\n{col}:")
    vc = df[col].value_counts()
    pct = (vc / len(df) * 100).round(1)
    display(pd.DataFrame({'count': vc, 'pct': pct}))

## 7. Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['stars'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title('Stars Distribution')
axes[0].set_xlabel('Stars')
axes[0].tick_params(axis='x', rotation=0)

axes[1].hist(df['review_count'], bins=50, color='steelblue', edgecolor='none')
axes[1].set_title('Review Count Distribution')
axes[1].set_xlabel('review_count')

plt.tight_layout()
plt.show()

In [ ]:
# Binary attribute prevalence bar chart
fig, ax = plt.subplots(figsize=(12, 5))
balance.plot(kind='bar', ax=ax, color='steelblue', edgecolor='none')
ax.set_title('Proportion of Restaurants with Each Binary Attribute')
ax.set_ylabel('Proportion')
ax.axhline(0.5, color='red', linestyle='--', linewidth=0.8, label='50%')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Categorical column distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, CATEGORICAL_COLS):
    df[col].value_counts().plot(kind='bar', ax=ax, color='steelblue', edgecolor='none')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 8. Ambience Correlation

In [ ]:
AMBIENCE_COLS = [c for c in BINARY_COLS if c.startswith('Ambience.')]

corr = df[AMBIENCE_COLS].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Ambience Flag Correlations')
plt.tight_layout()
plt.show()

## 9. Cleaning — Encode Categoricals & Fix Types

Issues to fix:
- `NoiseLevel`: ordered categorical (`quiet < average < loud < very_loud`); `unknown` → NaN
- `Alcohol`: unordered nominal (`none`, `beer_and_wine`, `full_bar`); `unknown` → NaN
- `RestaurantsAttire`: unordered nominal (`casual`, `dressy`, `formal`); `unknown` → NaN
- Binary float columns → int
- No feature scaling or engineering yet

In [ ]:
print(f"Before cleaning: {len(df):,} rows")
df_clean = df.copy()

# Replace 'unknown' sentinels with NaN
for col in CATEGORICAL_COLS:
    df_clean[col] = df_clean[col].replace('unknown', np.nan)

print("NaNs after replacing 'unknown':")
print(df_clean[CATEGORICAL_COLS].isnull().sum())

In [ ]:
# Encode NoiseLevel as ordered integer (NaN stays NaN)
NOISE_ORDER = {'quiet': 0, 'average': 1, 'loud': 2, 'very_loud': 3}
df_clean['NoiseLevel'] = df_clean['NoiseLevel'].map(NOISE_ORDER)
print("NoiseLevel after encoding:")
print(df_clean['NoiseLevel'].value_counts(dropna=False).sort_index())

In [ ]:
# One-hot encode Alcohol (drop_first=False to keep all categories explicit)
alcohol_dummies = pd.get_dummies(df_clean['Alcohol'], prefix='Alcohol', dummy_na=False)
df_clean = pd.concat([df_clean.drop(columns='Alcohol'), alcohol_dummies], axis=1)
print("Alcohol one-hot columns:", alcohol_dummies.columns.tolist())

In [ ]:
# One-hot encode RestaurantsAttire
attire_dummies = pd.get_dummies(df_clean['RestaurantsAttire'], prefix='Attire', dummy_na=False)
df_clean = pd.concat([df_clean.drop(columns='RestaurantsAttire'), attire_dummies], axis=1)
print("Attire one-hot columns:", attire_dummies.columns.tolist())

In [ ]:
# Cast binary float columns to int
# (NoiseLevel is now float with NaN, alcohol/attire dummies are bool — cast those to int too)
bool_cols = df_clean.select_dtypes(include='bool').columns.tolist()
df_clean[bool_cols] = df_clean[bool_cols].astype(int)

df_clean[BINARY_COLS] = df_clean[BINARY_COLS].astype(int)
print("Binary cols cast to int.")

print("\nFinal dtypes:")
print(df_clean.dtypes)

In [ ]:
# Remove duplicates (full rows, then by business_id)
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Dropped {before - len(df_clean)} full duplicate rows")

before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset='business_id', keep='first')
print(f"Dropped {before - len(df_clean)} duplicate business_ids")

print(f"\nAfter cleaning: {len(df_clean):,} rows x {df_clean.shape[1]} columns")

In [ ]:
# Final sanity check
assert df_clean['business_id'].duplicated().sum() == 0, "Duplicate business_ids remain!"
assert df_clean[BINARY_COLS].isin([0, 1]).all().all(), "Binary cols have unexpected values!"
assert df_clean['stars'].between(1.0, 5.0).all(), "Stars out of range!"
print("All assertions passed — data is clean and ML-ready.")
df_clean.head(3)

In [ ]:
OUT_PATH = r"C:\Users\ryanm\Documents\coding_projects\restaurant_recommendation\data\processed\yelp_ml_ready.csv"
df_clean.to_csv(OUT_PATH, index=False)
print(f"Saved to: {OUT_PATH}")
print(f"Final shape: {df_clean.shape}")